# Assignment 4 — Bayesian Decision Theory
**ENGS 3302 Pattern Recognition · Week 4** · Submit this notebook (`.ipynb`) on Moodle **with all cells run**
(*Runtime → Restart and run all*, then *File → Download → .ipynb*).

**How it is marked (about 1% of the course):**
**1** = the notebook runs, every task has a real attempt, and every *explain in your own words* question is answered ·
**0.5** = more than half of the tasks are attempted · **0** = not submitted, empty, or copied.
In the in-person sessions you may be asked to explain part of your work for two minutes; if you cannot explain it, the mark is 0.
Late work: accepted up to 48 hours after the deadline without penalty, not after.

In [1]:
# Run this cell first. It loads the libraries, a helper that checks your answers, and the penguin data.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from sklearn.model_selection import train_test_split

def check(label, got, expected, tol=1e-2):
    """Prints a green tick if `got` is close to `expected`, otherwise a red cross."""
    try:
        if isinstance(expected, str):
            ok = str(got) == expected
        else:
            ok = got is not None and np.allclose(np.asarray(got, dtype=float),
                                                 np.asarray(expected, dtype=float), atol=tol)
    except Exception:
        ok = False
    print(f"✅ {label}" if ok else f"❌ {label}: expected {expected}, got {got}")

# Palmer Penguins (Horst, Hill & Gorman 2020, CC0): 344 penguins of 3 species, measured in mm and g.
# Uses penguins.csv from the Moodle zip if it is next to the notebook, otherwise downloads it.
PENGUIN_FEATURES = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
def load_penguins():
    """Returns the cleaned table: 342 penguins with all four measurements."""
    try:
        df = pd.read_csv("penguins.csv")
    except FileNotFoundError:
        df = pd.read_csv("https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv")
    return df.dropna(subset=PENGUIN_FEATURES).reset_index(drop=True)   # 2 penguins have no measurements

peng = load_penguins()
print("Ready.", len(peng), "penguins")

Ready. 342 penguins


### Task 1 — A day with few sea bass
Only **30%** of today's catch are sea bass (lightness models as in the Notes: sea bass $N(7,1)$, salmon $N(4,1)$).
For $x = 4.5,\ 5.5,\ 6.5$ print the two likelihoods, the posterior $P(\text{sea bass}\mid x)$ and the decision (minimum error).

In [2]:
from scipy.stats import norm
prior = [0.3, 0.7] # sea bass, salmon


# Step 2: Define lightness values
x_values = [4.5, 5.5, 6.5]

# Step 3: Calculate likelihoods

for x in x_values:

    likelihood_SB = norm.pdf(x, loc=7, scale=1)
    likelihood_Salmon = norm.pdf(x, loc=4, scale=1)

    print("x =", x)
    print("Sea Bass Likelihood =", likelihood_SB)
    print("Salmon Likelihood =", likelihood_Salmon)

    # Step 4: Calculate posterior
    lik = [likelihood_SB * prior[0], likelihood_Salmon * prior[1]]
    joint = [lik[0] * prior[0], lik[1] * prior[1]]

    post = [j / sum(joint) for j in joint]
    posterior_SB = post[0]
    posterior_Salmon = post[1]

    # Step 5: Make the decision
    print(f"\nFor x = {x}")
    print(f"Sea Bass Likelihood = {likelihood_SB:.6f}")
    print(f"Salmon Likelihood = {likelihood_Salmon:.6f}")
    print(f"Posterior P(Sea Bass | x) = {posterior_SB:.6f}")

    if posterior_SB > 0.5:
        print("Decision = Sea Bass")
    else:
        print("Decision = Salmon")


x = 4.5
Sea Bass Likelihood = 0.01752830049356854
Salmon Likelihood = 0.3520653267642995

For x = 4.5
Sea Bass Likelihood = 0.017528
Salmon Likelihood = 0.352065
Posterior P(Sea Bass | x) = 0.009062
Decision = Salmon
x = 5.5
Sea Bass Likelihood = 0.12951759566589174
Salmon Likelihood = 0.12951759566589174

For x = 5.5
Sea Bass Likelihood = 0.129518
Salmon Likelihood = 0.129518
Posterior P(Sea Bass | x) = 0.155172
Decision = Salmon
x = 6.5
Sea Bass Likelihood = 0.3520653267642995
Salmon Likelihood = 0.01752830049356854

For x = 6.5
Sea Bass Likelihood = 0.352065
Salmon Likelihood = 0.017528
Posterior P(Sea Bass | x) = 0.786743
Decision = Sea Bass


**Explain in your own words (2–3 sentences):** At $x=5.5$ the two likelihoods are equal. Why is the posterior exactly 0.3, and why do we decide salmon?

*Your answer:* At x = 5.5, the two likelihoods are equal. Therefore, the posterior probabilities depend only on the prior probabilities. Since the prior probability of Sea Bass is 0.3 , We decide Salmon because it has the higher posterior probability (0.7), which minimizes the classification error.

### Task 2 — The best threshold for this day
Keep the 30% day. For thresholds `t = np.linspace(4, 7, 3001)` compute the total error
$0.3\cdot P(x<t\mid\text{sea bass}) + 0.7\cdot P(x>t\mid\text{salmon})$ and find the threshold with the smallest error. Print that threshold, its error, and the error of $t=5.5$.

In [3]:
import numpy as np
from scipy.stats import norm

# Step 1: Define thresholds
t = np.linspace(4, 7, 3001)

# Step 2: Calculate total error
err = 0.3 * norm.cdf(t, 7, 1) + 0.7 * (1 - norm.cdf(t, 4, 1))

# Step 3: Find the minimum error
i = np.argmin(err)

best_threshold = t[i]
best_error = err[i]

# Step 4: Calculate error at t = 5.5
error_5_5 = 0.3 * norm.cdf(5.5, 7, 1) + 0.7 * (1 - norm.cdf(5.5, 4, 1))

# Step 5: Print results
print("Best threshold:", best_threshold)
print("Minimum error:", best_error)
print("Error at t = 5.5:", error_5_5)

Best threshold: 5.782
Minimum error: 0.05964582701855205
Error at t = 5.5: 0.06680720126885809


**Explain in your own words (2–3 sentences):** Did the best threshold move left or right of 5.5? Explain why, using the two shaded areas of Figure 2 in the Notes.

*Your answer:*The best threshold moved to the right of 5.5 (approximately 5.782). Moving the threshold to the right decreases the blue shaded area (sea bass called salmon) but increases the orange shaded area (salmon called sea bass). Since salmon has a higher prior probability (70%) than sea bass (30%), the threshold shifts to the right to minimize the total weighted classification error.

### Task 3 — Fewer mistakes or lower cost?
Back to 60% sea bass. Now calling a sea bass "salmon" costs **5**, and calling a salmon "sea bass" costs 1.
(a) Find the minimum-risk threshold (decide sea bass when $5\,P(\omega_1\mid x) > P(\omega_2\mid x)$) with a grid search.
(b) For this threshold and for the minimum-error threshold 5.365, print the total error and the average cost per fish
(cost $= 5\cdot 0.6\cdot P(x<t\mid\text{sea bass}) + 1\cdot 0.4\cdot P(x>t\mid\text{salmon})$).

In [4]:
import numpy as np
from scipy.stats import norm

# Step 1: Define thresholds
t = np.linspace(1, 10, 9001)

# Step 2: Calculate risk
risk = 5 * 0.6 * norm.cdf(t, 7, 1) + 0.4 * (1 - norm.cdf(t, 4, 1))

# Step 3: Find minimum risk threshold
i = np.argmin(risk)
best_t = t[i]

# Step 4: Define thresholds for comparison
thresholds = [best_t, 5.365]

# Step 5: Calculate error and cost
for threshold in thresholds:

    error = 0.6 * norm.cdf(threshold, 7, 1) + 0.4 * (1 - norm.cdf(threshold, 4, 1))

    cost = 5 * 0.6 * norm.cdf(threshold, 7, 1) + 0.4 * (1 - norm.cdf(threshold, 4, 1))

    print("Threshold:", threshold)
    print("Total Error:", error)
    print("Average Cost:", cost)
    print("----------------")

Threshold: 4.828
Total Error: 0.09049079885402594
Average Cost: 0.1263175943542986
----------------
Threshold: 5.365
Total Error: 0.06506532429060818
Average Cost: 0.18752418143757973
----------------


**Explain in your own words (3–4 sentences):** Which threshold makes more mistakes? Which one costs less? Give one real system (not fish) where you would choose the minimum-risk rule, and say which mistake is the expensive one.

*Your answer:* The minimum-error threshold (5.365) makes fewer mistakes, with a total error of 6.51%. However, the minimum-risk threshold (4.828) costs less, with an average cost of 0.126318 compared to 0.187524.

A real example is a bank fraud detection system. Missing a fraudulent transaction is more expensive than incorrectly flagging a legitimate transaction as suspicious. Therefore, I would choose the minimum-risk rule because it reduces the cost of the more expensive mistake.

### Task 4 — A Bayes classifier for three species
Build the classifier of Lab 1.5 with the feature **`bill_length_mm`** instead of the flipper length (same split: `test_size=0.3, random_state=0, stratify`).
Print the test accuracy **with** the priors and **with equal priors** (1/3 each).

In [5]:
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score

# Step 1: Load dataset
df = sns.load_dataset("penguins")

# Step 2: Remove missing values
df = df.dropna(subset=["bill_length_mm", "species"])

# Step 3: Select feature and target
X = df[["bill_length_mm"]]
y = df["species"]

# Step 4: Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=0,
    stratify=y
)

# Step 5: Classifier with original priors
model1 = GaussianNB()
model1.fit(X_train, y_train)

pred1 = model1.predict(X_test)
acc1 = accuracy_score(y_test, pred1)

# Step 6: Classifier with equal priors
model2 = GaussianNB(priors=[1/3, 1/3, 1/3])
model2.fit(X_train, y_train)

pred2 = model2.predict(X_test)
acc2 = accuracy_score(y_test, pred2)

# Step 7: Print results
print("Accuracy with original priors:", acc1)
print("Accuracy with equal priors:", acc2)

Accuracy with original priors: 0.7961165048543689
Accuracy with equal priors: 0.7281553398058253


**Explain in your own words (2–3 sentences):** With the flipper length (Lab 1.5) the priors did not change the accuracy. With the bill length they do. Why can the priors matter more for one feature than for another? (Hint: which species overlap on each feature, and how common is each of them?)

*Your answer:* With flipper length, the three species are relatively well separated, so changing the priors does not significantly affect classification accuracy
However, with bill length, Chinstrap and Gentoo overlap considerably because their average bill lengths are similar. Also, the species have different numbers of samples. Therefore, changing the priors affects the classification decisions and can change the test accuracy.

### Task 5 — Likelihood or posterior?
**Explain in your own words (3–4 sentences):** A classmate writes: *"At 195 mm the Chinstrap curve is the highest, so the penguin is most probably a Chinstrap."*
Use Worked example 4.7 to explain what is wrong. Which number answers "most probably", and what does the classmate need to add?

*Your answer:* The classmate is confusing likelihood with posterior probability. Although Chinstrap has the highest likelihood (0.0560), Adelie has a higher prior probability (0.44). After multiplying the likelihood by the prior and normalizing, the posterior probability of Adelie is 0.642, which is the highest. Therefore, the penguin is most probably an Adelie. The classmate needs to consider the prior probabilities, not just the likelihood.

### AI use
Did you use an AI tool for this assignment? If yes: which tool, for which task, and what did you change yourself? (If no, write "No".)

*Your answer:* Yes, I used the free version of ChatGPT as a learning aid during this assignment. I used it to help me understand the concepts, write and debug Python code, and clarify topics that I initially found difficult. I reviewed the explanations, tested the code, corrected errors, and made sure I understood the steps and results myself. The final work reflects my own understanding and the changes I made based on what I learned.